<a href="https://colab.research.google.com/github/Ashu-42/quant_research_crypto_volatility_forecasting/blob/quant_DL_ashu/notebooks/01_Data_Creation_QP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Imports

In [1]:
import requests
import pandas as pd
import numpy as np
import time
import os
from google.colab import drive
from pathlib import Path
import shutil
import json

### Binance API

In [2]:
# Binance API address

base_url = "https://data-api.binance.vision"
endpoint = "/api/v3/klines"

url = base_url + endpoint

print(url)

https://data-api.binance.vision/api/v3/klines


### Structured - Paginated data pull

In [3]:
# converting to pandas DF

columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "close_time",
    "quote_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
]

numeric_columns = [
    "open",
    "high",
    "low",
    "close",
    "base_volume",
    "quote_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume"
]

In [5]:
def interval_to_milliseconds(interval):

    unit = interval[-1]
    value = int(interval[:-1])

    if unit == "m":
        return value * 60 * 1000

    elif unit == "h":
        return value * 60 * 60 * 1000

    elif unit == "d":
        return value * 24 * 60 * 60 * 1000

    elif unit == "w":
        return value * 7 * 24 * 60 * 60 * 1000

    else:
        raise ValueError(
            f"Unsupported interval: {interval}"
        )

In [6]:
def download_raw_klines(symbol, interval, start_date, end_date, limit=1000):

    start_timestamp = pd.Timestamp(
        start_date,
        tz="UTC"
    )

    end_timestamp = pd.Timestamp(
        end_date,
        tz="UTC"
    )

    current_start_ms = int(
        start_timestamp.timestamp() * 1000
    )

    end_time_ms = int(
        end_timestamp.timestamp() * 1000
    )

    interval_ms = interval_to_milliseconds(
        interval
    )

    all_data = []

    while current_start_ms < end_time_ms:

        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": current_start_ms,
            "endTime": end_time_ms,
            "limit": limit
        }

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        batch = response.json()

        if len(batch) == 0:
            break

        all_data.extend(batch)

        last_open_time = batch[-1][0]

        current_start_ms = (
            last_open_time + interval_ms
        )

        print(
            f"Downloaded {len(all_data)} rows "
            f"for {symbol}"
        )

        time.sleep(0.2)

    return all_data

In [7]:
def basic_dp(df, numeric_columns, asset, symbol):

  df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
  df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)

  df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric)
  df = df.drop(columns=["ignore"])

  df["asset"] = asset
  df["symbol"] = symbol

  df = df[
    [
        "open_time",
        "close_time",
        "asset",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "base_volume",
        "quote_volume",
        "number_of_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume"
    ]
  ]

  return df

In [8]:
 # Getting all the relevant data togethor

def get_crypto_data(symbol, asset, columns, numeric_columns, start_date, end_date, interval="1d", limit=1000):

    raw_data = download_raw_klines(symbol, interval, start_date, end_date, limit=limit)

    df = pd.DataFrame(
        raw_data,
        columns=columns
    )

    df = (
    df.sort_values("open_time")
      .drop_duplicates(subset=["open_time"])
      .reset_index(drop=True)
    )

    df = basic_dp(df, numeric_columns, asset, symbol)

    return df


In [9]:
# Checking the function

assets = {
    "BTC": "BTCUSDT",
    "ETH": "ETHUSDT",
    "SOL": "SOLUSDT",
    "XRP": "XRPUSDT"
}

start_date = "2017-01-01"
end_date = "2026-08-01"

crypto_dfs = []

for asset, symbol in assets.items():
  crypto_dfs.append(get_crypto_data(symbol, asset, columns, numeric_columns, start_date, end_date, interval="1d", limit=1000))


crypto_df = pd.concat(
        crypto_dfs,
    ignore_index=True
)

crypto_df = (
    crypto_df
    .sort_values(
        ["open_time", "asset"]
    )
    .reset_index(drop=True)
)

Downloaded 1000 rows for BTCUSDT
Downloaded 2000 rows for BTCUSDT
Downloaded 3000 rows for BTCUSDT
Downloaded 3272 rows for BTCUSDT
Downloaded 1000 rows for ETHUSDT
Downloaded 2000 rows for ETHUSDT
Downloaded 3000 rows for ETHUSDT
Downloaded 3272 rows for ETHUSDT
Downloaded 1000 rows for SOLUSDT
Downloaded 2000 rows for SOLUSDT
Downloaded 2182 rows for SOLUSDT
Downloaded 1000 rows for XRPUSDT
Downloaded 2000 rows for XRPUSDT
Downloaded 3000 rows for XRPUSDT
Downloaded 3012 rows for XRPUSDT


### Saving and Sharing the DF

In [10]:
temporary_path = "/content/crypto_daily_raw.parquet"

crypto_df.to_parquet(
    temporary_path,
    index=False
)

file_size_mb = os.path.getsize(temporary_path) / (1024 ** 2)

print(f"File size: {file_size_mb:.2f} MB")

File size: 0.91 MB


In [11]:
test_df = pd.read_parquet(temporary_path)

print(test_df.shape)
print(test_df.dtypes)
assert len(test_df) == len(crypto_df)

(11738, 13)
open_time                 datetime64[ns, UTC]
close_time                datetime64[ns, UTC]
asset                                  object
symbol                                 object
open                                  float64
high                                  float64
low                                   float64
close                                 float64
base_volume                           float64
quote_volume                          float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
dtype: object


#### Storing in Gdrive

In [12]:
# Accessing the google drive

drive.mount("/content/drive")

Mounted at /content/drive


In [13]:
shared_project_dir = Path(
    "/content/drive/MyDrive/Quant Research"
)

print(shared_project_dir.exists())

True


In [14]:
data_dir = shared_project_dir / "data"
raw_dir = data_dir / "raw"

raw_dir.mkdir( parents=True, exist_ok=True)

print(raw_dir)

filename = "crypto_binance_1d_btc_eth_sol_xrp_raw.parquet"

shared_file_path = (raw_dir / filename)

shutil.copy2(temporary_path, shared_file_path)

print("Saved to:")
print(shared_file_path)

/content/drive/MyDrive/Quant Research/data/raw
Saved to:
/content/drive/MyDrive/Quant Research/data/raw/crypto_binance_1d_btc_eth_sol_xrp_raw.parquet


In [15]:
# checking if export was correct

drive_df = pd.read_parquet(shared_file_path)

print(drive_df.shape)

assert len(drive_df) == len(crypto_df)

(11738, 13)


In [16]:
print(shared_file_path.exists())
print(
    f"{shared_file_path.stat().st_size / 1024**2:.2f} MB"
)

True
0.91 MB


### Manifest Creation

In [17]:
summary = (
    crypto_df.groupby("asset")
    .agg(
        rows=("open_time", "size"),
        first_timestamp=("open_time", "min"),
        last_timestamp=("open_time", "max")
    )
    .reset_index()
)

summary["first_timestamp"] = (
    summary["first_timestamp"].astype(str)
)

summary["last_timestamp"] = (
    summary["last_timestamp"].astype(str)
)

manifest = {
    "dataset_name": filename,
    "source": "Binance Spot API",
    "interval": "1d",
    "assets": list(assets.keys()),
    "symbols": assets,
    "total_rows": int(len(crypto_df)),
    "columns": crypto_df.columns.tolist(),
    "asset_summary": summary.to_dict(
        orient="records"
    )
}

In [18]:
manifest_path = (
    shared_project_dir / "manifests" /
    "crypto_binance_1d_btc_eth_sol_xrp_raw_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        manifest,
        file,
        indent=2
    )

print(manifest_path)

/content/drive/MyDrive/Quant Research/manifests/crypto_binance_1d_btc_eth_sol_xrp_raw_manifest.json


### Basic QC

In [19]:
crypto_df.groupby("asset").size()

,0
asset,
BTC,3272
ETH,3272
SOL,2182
XRP,3012


In [20]:
crypto_df.groupby("asset").agg(
    first_date=("open_time", "min"),
    last_date=("open_time", "max"),
    rows=("open_time", "size")
)

,first_date,last_date,rows
asset,,,
BTC,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272
ETH,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272
SOL,2020-08-11 00:00:00+00:00,2026-08-01 00:00:00+00:00,2182
XRP,2018-05-04 00:00:00+00:00,2026-08-01 00:00:00+00:00,3012


In [21]:
print(crypto_df.shape)
print()
print(crypto_df.dtypes)

(11738, 13)

open_time                 datetime64[ns, UTC]
close_time                datetime64[ns, UTC]
asset                                  object
symbol                                 object
open                                  float64
high                                  float64
low                                   float64
close                                 float64
base_volume                           float64
quote_volume                          float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
dtype: object


In [22]:
crypto_df.duplicated(
    subset=["asset", "open_time"]
).sum()

np.int64(0)

In [23]:
crypto_df.isna().sum()

,0
open_time,0
close_time,0
asset,0
symbol,0
open,0
high,0
low,0
close,0
base_volume,0
quote_volume,0


In [24]:
crypto_df.head()

,open_time,close_time,asset,symbol,open,high,low,close,base_volume,quote_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2017-08-17 00:00:00+00:00,2017-08-17 23:59:59.999000+00:00,BTC,BTCUSDT,4261.48,4485.39,4200.74,4285.08,795.150377,3.454770e+06,3427,616.248541,2.678216e+06
1,2017-08-17 00:00:00+00:00,2017-08-17 23:59:59.999000+00:00,ETH,ETHUSDT,301.13,312.18,298.00,302.00,7030.710340,2.154655e+06,4522,6224.589990,1.908705e+06
2,2017-08-18 00:00:00+00:00,2017-08-18 23:59:59.999000+00:00,BTC,BTCUSDT,4285.08,4371.52,3938.77,4108.37,1199.888264,5.086958e+06,5233,972.868710,4.129123e+06
3,2017-08-18 00:00:00+00:00,2017-08-18 23:59:59.999000+00:00,ETH,ETHUSDT,302.00,311.79,283.94,293.96,9537.846460,2.858947e+06,5658,7452.435420,2.240813e+06
4,2017-08-19 00:00:00+00:00,2017-08-19 23:59:59.999000+00:00,BTC,BTCUSDT,4108.37,4184.69,3850.00,4139.98,381.309763,1.549484e+06,2153,274.336042,1.118002e+06
